先用Random Forest做一遍——计算其特征的man的预测率

In [130]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import seaborn.objects as so
from matplotlib.pyplot import figure
from statsmodels.tsa.vector_ar.var_model import test_normality
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_curve, auc, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeRegressor

from scipy.stats import trim_mean
from scipy.stats import median_abs_deviation
from scipy.stats import spearmanr

导入数据：

In [131]:
train = pd.read_csv("titanic/train.csv")
test = pd.read_csv("titanic/test.csv")
truth = pd.read_csv("titanic/truth.csv")


test['Survived'] = np.nan

data = pd.concat([train, test], ignore_index=True, sort=False)

具有缺失值的特征有：Age, Fare, Cabin, Embarked

先填充Cabin和Embarked，再用DecisionTreeRegressor填充Age和Fare

In [132]:
# 填充Embarked
data['Embarked'] = data['Embarked'].fillna('S')

In [133]:
# 填充Cabin
data['Deck'] = data['Cabin'].apply(lambda x: 'M' if pd.isna(x) else x[0])
data.loc[data[data['Deck']=='T'].index, 'Deck'] = 'A'
data.replace({'Deck':['A','B','C']},'ABC', inplace=True)
data.replace({'Deck':['D','E']}, 'DE', inplace=True)
data.replace({'Deck':['F','G']},'FG', inplace=True)

In [134]:
data.drop(columns='Cabin', axis=1, inplace=True)

构建Title字段

In [135]:
data['Title'] = 'man'
data.loc[data['Name'].str.contains('Master', na=False),'Title'] = 'boy'
data.loc[data['Sex']=='female','Title'] = 'woman'

In [136]:
null_index = data[data['Age'].isnull()].index

In [137]:
null_fare = data[data['Fare'].isnull()].index

In [138]:
data['Title_Encode'] = data['Title'].map({'man':0,'woman':1,'boy':2})
data['Embarked_code'] = data['Embarked'].map({'S':0,'Q':1,'C':2})
data['Sex_code'] = data['Sex'].map({'male':0, 'female':1})

In [139]:
# def impute_with_tree(df, target, features):
#     df = df.copy()
#
#     # 用中位数进行填充
#     for f in features:
#         if df[f].isnull().any():
#             df[f] = df[f].fillna(df[f].median())
#     x_train = df[df[target].notnull()][features]
#     x_test = df[df[target].isnull()][features]
#     if len(x_test) == 0:
#         return df[target]
#     y_train = df[df[target].notnull()][target]
#     model = DecisionTreeRegressor(random_state=0)
#     model.fit(x_train, y_train)
#     df.loc[df[target].isnull(), target]  = model.predict(x_test)
#     return df[target]
#
# data['Age'] = impute_with_tree(data, 'Age', ['Title_Encode', 'Parch','SibSp','Pclass'])
#
# # 填 Fare：用 Title_code, Pclass, Embarked_code, Sex_code, Age
# data['Fare'] = impute_with_tree(data, 'Fare', ['Title_Encode','Pclass','Embarked_code','Sex_code','Age'])

data['Age'] = data.groupby(['Pclass','Sex'], dropna=False)['Age'].transform(lambda x: x.fillna(x.median()))

cabin_fare = data[(data['Pclass']==3)&(data['Embarked']=='S')&(data['SibSp']==0)&(data['Parch']==0)]

data['Fare'] = data['Fare'].fillna(cabin_fare['Fare'].median())

In [140]:
data['FamilySize'] = data['Parch'] + data['SibSp'] + 1
data['Ticket'] = data['Ticket'].astype(str).str.strip()

In [141]:
data['TicketFreq'] = data.groupby('Ticket')['Ticket'].transform('count')

In [142]:
data['Surname'] = data['Name'].str.split(',').str[0].str.strip()

In [143]:
train_ft = data[data['PassengerId']<=891][['PassengerId','Surname','Ticket','Survived']]
test_ft = data[data['PassengerId']>891][['PassengerId','Surname','Ticket']]

common_families = set(train_ft['Surname'].dropna()).intersection(test_ft['Surname'].dropna())
common_tickets = set(train_ft['Ticket'].dropna()).intersection(test_ft['Ticket'].dropna())

family_survived = train_ft.groupby('Surname').agg(
    survival_rate = ('Survived','mean'),
    members = ('Survived','size')
)

family_rate = family_survived.loc[(family_survived['members']>1)&(family_survived.index.isin(common_families)), 'survival_rate'].to_dict()

ticket_survived = train_ft.groupby('Ticket').agg(
    survival_rate = ('Survived','mean'),
    members = ('Survived','size')
)

ticket_rate = ticket_survived.loc[(ticket_survived.index.isin(common_tickets))&(ticket_survived['members']>1), 'survival_rate'].to_dict()

In [144]:
mean_survival_rate = np.mean(train['Survived'])
# Family
train_ft['FamilySurvivalRate'] = train_ft['Surname'].map(family_rate).fillna(mean_survival_rate)
train_ft['FamilySurvivalRate_NA'] = train_ft['Surname'].map(family_rate).notna().astype(int)

test_ft['FamilySurvivalRate'] = test_ft['Surname'].map(family_rate).fillna(mean_survival_rate)
test_ft['FamilySurvivalRate_NA'] = test_ft['Surname'].map(family_rate).notna().astype(int)

# Ticket
train_ft['TicketSurvivalRate'] = train_ft['Ticket'].map(ticket_rate).fillna(mean_survival_rate)
train_ft['TicketSurvivalRate_NA'] = train_ft['Ticket'].map(ticket_rate).notna().astype(int)

test_ft['TicketSurvivalRate'] = test_ft['Ticket'].map(ticket_rate).fillna(mean_survival_rate)
test_ft['TicketSurvivalRate_NA'] = test_ft['Ticket'].map(ticket_rate).notna().astype(int)

In [145]:
for df in [train_ft, test_ft]:
    df['Survival_Rate'] = (df['FamilySurvivalRate']+df['TicketSurvivalRate'])/2
    df['Survival_Rate_NA'] = (df['FamilySurvivalRate_NA'] + df['TicketSurvivalRate_NA'])/2

In [146]:
ft_total = pd.concat([train_ft.drop('Survived', axis=1), test_df])
data = pd.merge(data, ft_total, on=['PassengerId','Surname','Ticket'])

In [147]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,Sex_code,FamilySize,TicketFreq,Surname,FamilySurvivalRate,FamilySurvivalRate_NA,TicketSurvivalRate,TicketSurvivalRate_NA,Survival_Rate,Survival_Rate_NA
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,...,0,2,1,Braund,0.383838,0,0.383838,0,0.383838,0.0
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,...,1,2,2,Cumings,0.383838,0,0.383838,0,0.383838,0.0
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,...,1,1,1,Heikkinen,0.383838,0,0.383838,0,0.383838,0.0
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,...,1,2,2,Futrelle,0.383838,0,0.383838,0,0.383838,0.0
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,...,0,1,1,Allen,0.383838,0,0.383838,0,0.383838,0.0


In [148]:
data['FareAdj'] = data['Fare']/data['TicketFreq']

In [149]:
data.loc[data['Title']=='man', 'Surname'] = 'noGroup'
data['SurnameFreq'] = data.groupby('Surname')['Surname'].transform('count')
data.loc[data['SurnameFreq']<=1,'Surname'] = 'noGroup'

# for i in data.index[(data['Title'] != "man") & (data['Surname'] == 'noGroup')]:
#     # 找到和这些乘客相同Ticket编号的乘客，标为对应乘客的Surname
#     same_ticket_surname = data.loc[data['Ticket'] == data.loc[i, 'Ticket'],"Surname"]
#     if not same_ticket_surname.empty:
#         data.loc[i, 'Surname'] = same_ticket_surname.iloc[0]

# data['Surname'] = data['Surname'].fillna('noGroup')
data['SurnameSurvival'] = np.nan
train_survival = data.iloc[:891].groupby("Surname")['Survived'].transform('mean')

data.loc[:890, 'SurnameSurvival'] = train_survival

类别变量编码

In [150]:
features = ['Embarked','Deck','Pclass']

data = pd.get_dummies(data, columns=features, dtype='int', drop_first=True)
data.head()

,PassengerId,Survived,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Title,...,FareAdj,SurnameFreq,SurnameSurvival,Embarked_Q,Embarked_S,Deck_DE,Deck_FG,Deck_M,Pclass_2,Pclass_3
0,1,0.0,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,man,...,7.25000,782,0.320675,0,1,0,0,1,0,1
1,2,1.0,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,woman,...,35.64165,1,0.320675,0,0,0,0,0,0,0
2,3,1.0,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,woman,...,7.92500,1,0.320675,0,1,0,0,1,0,1
3,4,1.0,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,woman,...,26.55000,1,0.320675,0,1,0,0,0,0,0
4,5,0.0,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,man,...,8.05000,782,0.320675,0,1,0,0,1,0,1


In [151]:
drop_cols = ['PassengerId','Name','SibSp','Parch','Ticket','Title_Encode','Embarked_code','Sex_code','Sex','Fare','SurnameSurvival','SurnameFreq','FamilySurvivalRate','FamilySurvivalRate_NA']

data.drop(drop_cols, axis=1, inplace=True)

In [152]:
data.head()

,Survived,Age,Title,FamilySize,TicketFreq,Surname,TicketSurvivalRate,TicketSurvivalRate_NA,Survival_Rate,Survival_Rate_NA,FareAdj,Embarked_Q,Embarked_S,Deck_DE,Deck_FG,Deck_M,Pclass_2,Pclass_3
0,0.0,22.0,man,2,1,noGroup,0.383838,0,0.383838,0.0,7.25000,0,1,0,0,1,0,1
1,1.0,38.0,woman,2,2,noGroup,0.383838,0,0.383838,0.0,35.64165,0,0,0,0,0,0,0
2,1.0,26.0,woman,1,1,noGroup,0.383838,0,0.383838,0.0,7.92500,0,1,0,0,1,0,1
3,1.0,35.0,woman,2,2,noGroup,0.383838,0,0.383838,0.0,26.55000,0,1,0,0,0,0,0
4,0.0,35.0,man,1,1,noGroup,0.383838,0,0.383838,0.0,8.05000,0,1,0,0,1,0,1


### 划分数据集

In [153]:
train_man_index = data.loc[(data.index<891)&(data['Title']=='woman')&(data['Surname']=='noGroup'),:].index
test_man_index = data.loc[(data.index>=891)&(data['Title']=='woman')&(data['Surname']=='noGroup'), :].index

# featuers = ['Age','Fare','FamilySize','TicketFreq']
X_train = data.loc[train_man_index,:].drop(['Survived','Title','Surname'], axis=1)
X_test = data.loc[test_man_index,:].drop(['Survived','Title','Surname'], axis=1)
y_train = data.loc[train_man_index, 'Survived']
y_test = truth.loc[truth['PassengerId'].isin(test_man_index+1),'Survived']

In [154]:
X_train.shape, X_test.shape

((174, 15), (88, 15))

In [155]:
# X_train

features = ['Age','FareAdj','Pclass_2','Pclass_3','Survival_Rate','Survival_Rate_NA']
X_train = X_train[features]
X_test = X_test[features]

### 数据归一化

In [156]:
scaler = StandardScaler()

features = ['Age','Fare','TicketFreq','FamilySize']
test_copy= X_test.copy()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 创建模型

In [157]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    penalty='l2',       # L2正则化
    class_weight='balanced',  # 处理类别不平衡
    solver='liblinear', # 小样本友好
    random_state=42
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)

0.7159090909090909

In [166]:
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
# ratio = np.sum(y_train == 0) / np.sum(y_train == 1)
model = XGBClassifier(
            objective='binary:logistic',  # 二分类逻辑回归
            use_label_encoder=False,      # 不使用过时的 label encoder
            # scale_pos_weight= ratio,
            eval_metric='error',          # 评估指标：错误率
            max_depth=5,                  # 树深度
            learning_rate=0.01,            # 学习率
            gamma=1,                    # 正则化参数 gamma
            colsample_bytree=1,           # 每棵树的特征采样比例
            min_child_weight=1,           # 叶子节点最小权重
            n_estimators=500,             # 树的数量
            random_state=42                # 随机种子
        )
model.fit(X_train, y_train)
p = model.predict_proba(X_test)[:,1]
y_pred_05 = (p > 0.5).astype(int)
print("默认阈值:")
print(classification_report(y_test, y_pred_05, digits=3))
print(accuracy_score(y_test, y_pred_05))

/Users/yujiewang/anaconda3/envs/Conda11/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [11:23:28] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


默认阈值:
              precision    recall  f1-score   support

           0      0.000     0.000     0.000        26
           1      0.687     0.919     0.786        62

    accuracy                          0.648        88
   macro avg      0.343     0.460     0.393        88
weighted avg      0.484     0.648     0.554        88

0.6477272727272727


In [164]:
y_pred = pd.DataFrame({'pred': y_pred})
y_pred['PassengerId'] = truth['PassengerId']
y_pred

ValueError: If using all scalar values, you must pass an index

In [160]:
# y_pred.to_csv('titanic/wcg_woman_submission.csv', index=False)

In [161]:
# wcg_sub = pd.read_csv("titanic/wcg_submissions.csv")
# wcg_sub